# Quickstart — oa_pipeline end-to-end

This tutorial runs the full eight-notebook ocean-acidification preprocessing
pipeline on a small, deterministic synthetic dataset and shows what the
analyst-facing output looks like.

**Time:** roughly 1–2 minutes once papermill is installed and the example
dataset is built.

**What you will see:**

1. Where the example dataset comes from (and how to regenerate it).
2. The full chain running stage-by-stage via the runner script.
3. The final `analysis_ready.csv` with its PASS / REVIEW / FAIL verdicts.
4. How to trace a flagged row back through the per-stage audit tables.

**Prerequisites:** This notebook assumes you have already installed the
pipeline:

```bash
pip install -e ".[dev]"      # editable install + pytest + papermill + parquet
```

It also assumes you are running this notebook from the *project root*
(the directory containing `run_pipeline.sh`, the eight stage notebooks,
and the `examples/` folder).


## 1. Locate the project root and example dataset

`__file__` is not defined in Jupyter, so we infer the project root from
the working directory. If you opened this notebook in Jupyter from the
project root, the next cell finds the example dataset.


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Jupyter's `pwd` is wherever you launched it. We assume the project root.
PROJECT_ROOT = Path.cwd().resolve()
print(f"Project root: {PROJECT_ROOT}")

EXAMPLE_XLSX = PROJECT_ROOT / "examples" / "example_data.xlsx"
EXAMPLE_GEN  = PROJECT_ROOT / "examples" / "make_example_data.py"

assert EXAMPLE_GEN.exists(), (
    f"Cannot find {EXAMPLE_GEN}. Are you running this notebook from the "
    f"project root (the folder containing run_pipeline.sh)?"
)
print(f"Example data generator: {EXAMPLE_GEN}")
print(f"Example dataset:        {EXAMPLE_XLSX} "
      f"({'exists' if EXAMPLE_XLSX.exists() else 'will be generated'})")


## 2. Generate (or regenerate) the example dataset

The example dataset is **synthetic but realistic**: 20 sample rows + 4 CRMs
+ 3 TRIS pH-standards of coastal-Ghana-style carbonate chemistry. Four of
the sample rows are deliberately broken so the pipeline has known issues
to flag — see the docstring of `make_example_data.py` for the full list.

The generator uses a fixed seed (`SEED = 20260516` — the EOI submission
date) so two runs produce byte-identical files.


In [ ]:
# Run the generator. Skip if the file already exists; force a rebuild by
# deleting examples/example_data.xlsx first.
if not EXAMPLE_XLSX.exists():
    result = subprocess.run(
        [sys.executable, str(EXAMPLE_GEN), "--out", str(EXAMPLE_XLSX)],
        capture_output=True, text=True, check=True,
    )
    print(result.stdout)
else:
    print(f"Re-using existing {EXAMPLE_XLSX.name}.")

# Take a peek
peek = pd.read_excel(EXAMPLE_XLSX, sheet_name="oa_data")
print(f"\nLoaded {len(peek)} rows, {len(peek.columns)} columns.")
print(f"\nRow categories:")
print(peek["crm_or_sample"].value_counts(dropna=False).to_string())
peek.head(3)


## 3. Run the full eight-notebook pipeline

The runner script `run_pipeline.sh` papermills each notebook in order,
wiring each stage's input to the previous stage's output. We put the
outputs under `outputs/quickstart/` so they don't collide with any
real-data runs you have alongside.


In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "quickstart"

# Wipe a previous quickstart run so the output reflects only this run.
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

# Drive the runner from the project root.
runner = PROJECT_ROOT / "run_pipeline.sh"
assert runner.exists(), f"Cannot find {runner}"

result = subprocess.run(
    ["bash", str(runner), str(EXAMPLE_XLSX), str(OUTPUT_ROOT)],
    cwd=PROJECT_ROOT,
    capture_output=True, text=True,
)

# Show the runner's summary lines (skip the verbose papermill internals).
last_lines = result.stdout.splitlines()[-15:]
print("\n".join(last_lines))
if result.returncode != 0:
    print(f"\nRunner exited with code {result.returncode}")
    print(result.stderr[-1500:])
    raise RuntimeError("Pipeline run failed -- see error output above.")


## 4. Load the final analysis-ready CSV

`analysis_ready.csv` is the analyst-facing deliverable. Every original
sample row is present (the pipeline never drops rows), with the original
columns plus all the audit columns and the verdict.


In [ ]:
final_csv = OUTPUT_ROOT / "oa_stage4_outputs" / "data" / "analysis_ready.csv"
assert final_csv.exists(), f"Stage 4 did not produce {final_csv}"

ar = pd.read_csv(final_csv)
print(f"analysis_ready.csv  rows: {len(ar)}   columns: {len(ar.columns)}")
print(f"\nVerdict distribution:")
print(ar["analysis_audit_status"].value_counts().to_string())


### Verdict counts

The four deliberately-broken sample rows in the generator should each
produce a non-PASS verdict:

| Row | Injected issue | Expected reason |
|------|----------------|-----------------|
| S005 | salinity = 50 (above sal_max 42) | `range_flag` |
| S007 | dropped `sample_id` | `missing_key` |
| S010 | DIC species sum off by 200 µmol/kg | `strict_dic_species_fail` |
| S015 | negative HCO₃ (physically impossible) | `strict_dic_species_fail` |

The remaining ~16 rows should be PASS. Confirm:


In [ ]:
reason_counts = pd.read_json(
    OUTPUT_ROOT / "oa_stage4_outputs" / "logs" / "manifest.json"
)["reason_code_counts"].dropna()
print(f"Reason codes that fired (Stage 4 manifest):")
print(reason_counts.to_string())


## 5. Drill into the deliberately-broken rows

Each broken row's verdict and reason codes:


In [ ]:
broken_tags = ["S005", "S007", "S010", "S015"]
key_cols = [c for c in [
    "sample_tag", "sample_id", "salinity",
    "co2aq_calc_umol_kg", "hco3_calc_umol_kg", "co3_calc_umol_kg",
    "dic_best_umol_kg",
    "analysis_audit_status", "analysis_audit_reason_codes",
] if c in ar.columns]

ar.loc[ar["sample_tag"].isin(broken_tags), key_cols]


## 6. Inspect a single row's lineage end-to-end

Pick the salinity-out-of-range row (S005). The pipeline keeps every flag
column from every stage so you can trace what fired where. Here are
the flag columns that fired for S005:


In [ ]:
row = ar.loc[ar["sample_tag"] == "S005"].iloc[0]
flag_cols = [c for c in ar.columns if c.startswith("flag_") or c.startswith("flag_audit_")]
fired = row[flag_cols][row[flag_cols] == True]
print(f"Flags that fired for S005:")
for col, val in fired.items():
    print(f"  {col}")

# Salinity range flag table (Stage 4 long-format output)
range_long = pd.read_csv(
    OUTPUT_ROOT / "oa_stage4_outputs" / "tables" / "range_flags_long.csv"
)
print(f"\nS005 range violations from range_flags_long.csv:")
range_long[range_long["sample_id"].astype(str).str.contains("OA-2024-005", na=False)]


## 7. Where to look next

Every stage produces the same four kinds of output under `outputs/quickstart/`:

```
outputs/quickstart/
    oa_prelim_data__qc_outputs/   # Notebook 02
    oa_stage1a_outputs/           # Notebook 04
    oa_stage1b_outputs/           # Notebook 05
    oa_stage2_outputs/            # Notebook 06
    oa_stage3_outputs/            # Notebook 07
    oa_stage4_outputs/            # Notebook 08 — the verdicts above
        data/      analysis_ready.csv  (and .parquet)
        tables/    range_flags_long.csv, dic_species_audit.csv, ...
        reports/   report.md   (human-readable, includes thresholds and flag counts)
        logs/      manifest.json (machine-readable provenance)
                   effective_config.json (full config applied)
```

When you have a real dataset:

1. Replace `EXAMPLE_XLSX` with your workbook path and re-run.
2. Inspect each stage's `reports/report.md` to see thresholds in use and
   per-flag counts.
3. The `outputs/quickstart/runs/<UTC-timestamp>/` folder under the project
   root contains a fully-executed copy of each notebook (with all cell
   outputs) for the audit trail.

For the design rationale of each stage, see the per-stage `.README.md`
files at the project root. For an "I want to change X, which file?" map,
see the top-level [README.md](README.md).


In [ ]:
# A quick look at the per-stage manifest contents:
print("Per-stage manifest excerpts:")
for stage_dir in sorted(OUTPUT_ROOT.iterdir()):
    manifest = stage_dir / "logs" / "manifest.json"
    if not manifest.exists():
        continue
    m = json.loads(manifest.read_text())
    rc = m.get("row_counts", {})
    print(f"\n  {stage_dir.name}/logs/manifest.json")
    print(f"    notebook:   {m.get('notebook', 'n/a')}")
    print(f"    rows:       {rc.get('n_rows', rc.get('staged_rows', 'n/a'))}")
    print(f"    parquet:    {m.get('parquet_written', 'n/a')}")
